In [4]:
import os
import re
import pickle as pkl
import ast
from collections import Counter
from typing import List, Tuple, Any
from tqdm import tqdm
import time

import pandas as pd
import numpy as np
import calibration as cal

import torch

from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')
pal = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [5]:
from llm_unsupervised_conf.metrics import *
from llm_unsupervised_conf.math_utils import stack_embeddings
from llm_unsupervised_conf.calibration import fit_predict_prob_models
from llm_unsupervised_conf.plots import plot_reliability, plot_avg_and_worstcase_by_method, plot_method_comparisons, METHODS_MAP_SHORT

In [6]:
from llm_unsupervised_conf.utils import maybe_add_verbal_conf_row, load_out_df, set_seed

In [7]:
def compute_consistency_from_df(df: pd.DataFrame) -> pd.Series:
    """
    train_df: columns ["id", "answer"], many rows per id.
    returns Series indexed by id: max frequency / count
    """
    counts = df.groupby("id")["answer"].value_counts(dropna=False)
    top = counts.groupby(level=0).max()
    denom = df.groupby("id")["answer"].size()
    return (top / denom).sort_index()


def metric_row(name, scores, correct, n_bins):
    return [
        name,
        get_ece1(scores, correct, n_bins=n_bins),
        get_ece2(scores, correct, n_bins=n_bins),
        get_mce(scores, correct, n_bins=n_bins),
        get_nll(scores, correct),
        brier_score_loss(correct, scores),
        roc_auc_score(correct, scores),
    ]


def run_exp(
    dataset,
    model,
    n=1000,
    k_train=100,
    temp_train=0.7,
    temp_test=0.6,
    prob_models=("ridge_clip",),   # <-- pass a list/tuple of the methods above
    include_verbal_conf=True,
    random_state=42,
    n_bins=12,
    embedding_text="question_response",
    drop_bad_rows=True,
    test_prop=0.6,
    verbose=False,
    include_alt_ablation=True,
    include_question_ablation=True,
    k_ablate=10
):
    """
    prob_models controls which embedding->prob models to include.
    Example:
      prob_models=["ridge_clip","ridge_logit","hgb_logit","mlp_logit","isotonic_on_ridge"]
    """

    set_seed(random_state)

    model_name = model.split("/")[-1]

    try:
        out_df, df_save_path = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text, drop_bad_rows)
    except:
        return None

    if include_alt_ablation:
        ablation_df, _ = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text="question_response_gemma", drop_bad_rows=True)
        out_df["alt_embeddings"] = ablation_df["embeddings"]

    if include_question_ablation:
        ablation_df, _ = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text="question", drop_bad_rows=True)
        out_df["question_embeddings"] = ablation_df["embeddings"]

    slim_df_path = f"../outputs/{model_name}/{dataset}/n_{n}_temp_{temp_train}_k_{k_train}_SLIM_train_df.pkl"
    with open(slim_df_path, "rb") as f:
        slim_df = pkl.load(f)

    sampled = slim_df.groupby("id", group_keys=False).sample(n=k_ablate, random_state=random_state)
    out_df["ablate_consistency"] = out_df["id"].map(compute_consistency_from_df(sampled))

    # --------------------
    # Split
    # --------------------
    train_df, test_df = train_test_split(
        out_df,
        test_size=test_prop,
        random_state=random_state,
        shuffle=True,
    )

    X_train = stack_embeddings(train_df, "embeddings")
    if k_ablate == 100:
        y_train = train_df["consistency"].astype(np.float32).to_numpy()  # target in [0,1]
    else:
        y_train = train_df["ablate_consistency"].astype(np.float32).to_numpy()  # target in [0,1]

    X_test = stack_embeddings(test_df, "embeddings")

    # These are for evaluation
    con_scores = test_df["consistency"].to_numpy(dtype=float)
    correct = test_df["correct"].to_numpy(dtype=int)

    logprobs = test_df["avg_logprobs"].to_numpy(dtype=float)
    ans_logprobs = test_df["ans_logprobs"].to_numpy(dtype=float)
    lp = np.exp(logprobs)
    alp = np.exp(ans_logprobs)

    if verbose:
        print(f"Results: (accuracy={correct.mean():.4f})")

    ########################
    # Baselines
    ########################
    score_sources = {
        "logprob": lp,
        "ans_logprob": alp,
        "oracle_sc": con_scores,
    }
    
    rows = [metric_row(name, scores, correct, n_bins) for name, scores in score_sources.items()]

    ########################
    # Ours
    ########################
    prob_models = list(prob_models) if prob_models is not None else []
    preds = fit_predict_prob_models(
        X_train=X_train,
        y_train_prob=y_train,
        X_test=X_test,
        methods=prob_models,
        random_state=random_state,
    )

    for method_name, pred_probs in preds.items():
        rows.append(metric_row(method_name, pred_probs, correct, n_bins))

    # --------------------
    # Pack + display
    # --------------------
    exp_df = pd.DataFrame(rows, columns=["Method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC"])
    exp_df["Model"] = model
    exp_df["Dataset"] = dataset
    exp_df["Accuracy"] = float(correct.mean())
    exp_df = exp_df[["Model", "Dataset", "Method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC", "Accuracy"]]

    return exp_df


In [8]:
def run_all_exps(
    models, 
    datasets, 
    exp_key=None, 
    n_trials=10, 
    n_bins=12, 
    test_prop=0.6,
    embedding_text="question_response",
    prob_models = ["ridge_clip","split_isotonic_on_ridge","split_isotonic_on_ridge_nrt"],
    k_ablate_list = [10, 20, 50, 100]
):

    full_df = []
    for dataset in datasets:
        print(f"[status] running {dataset}")
        start_time = time.time()
        for model in models:
            print(f"[status] running {model}")
            for k_ablate in k_ablate_list:
                for seed in range(n_trials):
                    df = run_exp(
                        dataset=dataset,
                        model=model,
                        random_state=seed,
                        n_bins=n_bins,
                        test_prop=test_prop,
                        embedding_text=embedding_text,
                        prob_models=prob_models,
                        k_ablate=k_ablate
                    )
                    if df is None:
                        continue
                    df["seed"] = seed
                    df["k_ablate"] = k_ablate
                    full_df.append(df)

                
        print("--- %s seconds ---" % (time.time() - start_time))
    
    full_df = pd.concat(full_df)


    display(full_df.head(5))

    full_df.to_csv("../results/n_samples_ablation.csv", index=False)

    return full_df



In [ ]:
models = [
    "Qwen/Qwen3-0.6B", 
    "Qwen/Qwen3-1.7B", 
    "Qwen/Qwen3-4B-Thinking-2507", 
    "Qwen/Qwen3-8B",
    "Qwen/Qwen3-14B",
    "nvidia/OpenReasoning-Nemotron-7B",
    "nvidia/Nemotron-Cascade-8B-Thinking",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
]
datasets = [
    "gsm8k",
    "polymath",
    "trivia_qa",
    "sciq",
    "webq"
]

exp_key = "n_samples"
n_trials = 50

k_ablate_list = [1, 5, 10, 20, 50, 100]

df = run_all_exps(models, datasets, exp_key, n_trials, k_ablate_list=k_ablate_list)

[status] running gsm8k
[status] running Qwen/Qwen3-0.6B


In [21]:
df = pd.read_csv("../results/n_samples_ablation.csv")

In [ ]:
show_cols = ["k_ablate", "ECE2", "Brier", "AUROC"]

comp_methods = ["split_isotonic_on_ridge_nrt"]
ablate_df = df[df["Method"].isin(comp_methods)]
ablate_df = ablate_df[show_cols].groupby(["k_ablate"]).mean().sort_values(["k_ablate", "ECE2"]).reset_index()
ablate_df = ablate_df[ablate_df["k_ablate"] > 1]

print("AVERAGE")
display(
    ablate_df
)

In [ ]:
print(ablate_df.to_latex(index=False, float_format="%.5f"))